# Fase 1: Data Engineering (La Explosión)

**Limpieza Estructural:** Eliminen las filas donde la columna Cuentas_Que_Sigue esté vacía (nula). Estandaricen la columna Usuario a MAYÚSCULAS y quítenle los espacios al inicio/final. Eliminen usuarios duplicados.

In [ ]:
import pandas as pd

df = pd.read_csv('export_api_influencers.csv')
df = df.dropna(subset=['Cuentas_Que_Sigue'])
df['Usuario'] = df['Usuario'].str.upper().str.strip()
df = df.drop_duplicates(subset='Usuario')

**La Explosión:** Separen los nombres de la columna Cuentas_Que_Sigue usando .str.split(','). Luego, utilicen la función mágica de Pandas .explode() sobre esa columna para aplanar la tabla y convertirla en una verdadera lista de Origen -> Destino.

In [ ]:
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(',')
aristas = df.explode('Cuentas_Que_Sigue')

**Purificación Post-Explosión:** Vuelvan a limpiar los textos de la columna destino (mayúsculas y sin espacios extra). Finalmente, eliminen los Self-Loops (Registros donde el Usuario aparezca siguiéndose a sí mismo).

In [ ]:
aristas['Cuentas_Que_Sigue'] = aristas['Cuentas_Que_Sigue'].str.upper().str.strip()
aristas = aristas[aristas['Usuario'] != aristas['Cuentas_Que_Sigue']]
aristas[['Usuario', 'Cuentas_Que_Sigue']]

# Fase 2: El Engaño de la Popularidad

Importen networkx y conviertan su tabla en un Grafo Dirigido (nx.DiGraph). Calculen la Centralidad de Entrada (nx.in_degree_centrality). Impriman a los 3 usuarios con el puntaje más alto.

In [ ]:
import networkx as nx

G = nx.from_pandas_edgelist(aristas, 'Usuario', 'Cuentas_Que_Sigue', create_using=nx.DiGraph)
in_deg = pd.Series(nx.in_degree_centrality(G)).sort_values(ascending=False)
in_deg.head(3)

**Pregunta:** A simple vista, ¿quién parece ser el rey de la red?

Parece ser AMATEUR_1, que tiene 0.72 de centralidad de entrada mientras GARNACHO apenas llega a 0.20.

# Fase 3: La Verdad Topológica (Las Métricas)

Calculen el PageRank (nx.pagerank). ¿Quién tiene ahora la verdadera autoridad?

In [ ]:
pr = pd.Series(nx.pagerank(G)).sort_values(ascending=False)
pr.head(3)

Parece ser AMATEUR_1, que tiene 0.72 de centralidad de entrada mientras GARNACHO apenas llega a 0.20.R

Conviertan el grafo a No Dirigido y calculen el Coeficiente de Clustering (nx.clustering). Impriman el valor exacto de AMATEUR_1 y el de GARNACHO.

In [ ]:
cl = nx.clustering(G.to_undirected())
print(cl['AMATEUR_1'], cl['GARNACHO'])

# Veredicto Final

**¿Quién compró bots y por qué?**

AMATEUR_1, ya que su clustering de 0 significa que sus 47 seguidores no tienen ninguna conexión entre ellos, y todo su PageRank proviene de cuentas sin seguidores.

**¿Quién es el influencer real y por qué?**

GARNACHO, ya que su clustering de 0.133 indica que sus seguidores están conectados entre sí, y con apenas 13 seguidores logra un PageRank de 0.084 parecido al de DE_PAUL porque lo siguen cuentas con autoridad.